In [1]:
import numpy as np
import json
import os
import glob
# def import_ktk():
#     # Patch np.Inf to np.inf
#     np.Inf = np.inf
    
#     # Import kineticstoolkit
#     import kineticstoolkit.lab as ktk
    
#     # Remove the patch
#     del np.Inf
    
#     return ktk

# # Use the custom import function
# ktk = import_ktk()
import kineticstoolkit.lab as ktk
# Set an interactive backend, not required if already enabled in Spyder
%matplotlib qt5

In [2]:
# json_folder = '/home/mohan/nas_drive/methods/ped_gen/waymo/processed/dataset_waymo_pretrain'
# json_path = os.path.join(json_folder, '*.json')
# json_files = glob.glob(json_path)
# json_file_names = [os.path.basename(f) for f in json_files]
# print(json_files)
# with open(json_files[1], 'r') as f:
#     json_data = json.load(f)
# print('loaded file:', json_file_names[1])

In [3]:
json_local = "/home/erik/NAS/methods/diffusion_gen/data/diffusion/ava/train/0001_33.json"
with open(json_local, 'r') as f:
    json_data_local = json.load(f)
print('loaded local file:', json_local)

json_data = json_data_local

loaded local file: /home/erik/NAS/methods/diffusion_gen/data/diffusion/ava/train/0001_33.json


In [4]:
np.array(json_data['ego_in_ped_frame']).shape

(186, 3)

In [5]:
joint_names = {

'Pelvis': 'Hips', #0
'L_Hip' : 'LeftThigh', #1
'R_Hip': 'RightThigh', #2
'Spine1': 'SpineMid', #3
'L_Knee': 'LeftLeg', #4
'R_Knee': 'RightLeg', #5
'Spine2': 'Chest', #6
'L_Ankle': 'LeftFoot', #7
'R_Ankle': 'RightFoot', #8
'Spine3': 'Neck', #9
'L_Foot': 'LeftToe', #10
'R_Foot': 'RightToe', #11
'Neck': 'Head', #12
'L_Collar': 'LeftShoulder', #13
'R_Collar': 'RightShoulder', #14
'Head': 'HeadTop', #15
'L_Shoulder': 'LeftArm', #16
'R_Shoulder': 'RightArm', #17
'L_Elbow': 'LeftForearm', #18
'R_Elbow': 'RightForearm', #19
'L_Hand': 'LeftHand', #20
'R_Hand': 'RightHand' #21
    # ... other new names and their corresponding original keys ...
}

def add_fourth_column(time_series, value=1.0):
    """
    Add a 4th column to the 'data' array in a TimeSeries object and fill it with a specified value.
    
    Args:
    time_series (ktk.TimeSeries): The TimeSeries object to modify.
    value (float): The value to fill the new column with. Default is 1.0.
    
    Returns:
    ktk.TimeSeries: The modified TimeSeries object.
    """
    current_data = time_series.data['data']
    new_data = np.pad(current_data, ((0, 0), (0, 0), (0, 1)), 
                      mode='constant', constant_values=value)
    
    # Create a new TimeSeries object with the updated data
    new_time_series = time_series.copy()
    new_time_series.data['data'] = new_data
    
    return new_time_series

def restructure_timeseries(ts, joint_names):
    """
    Restructure the TimeSeries object to have separate entries for each joint with custom names.
    
    Args:
    ts (ktk.TimeSeries): The original TimeSeries object.
    joint_names (dict): Dictionary mapping original joint names to new names.
    
    Returns:
    ktk.TimeSeries: A new TimeSeries object with restructured data and custom joint names.
    """
    new_ts = ktk.TimeSeries()
    new_ts.time = ts.time
    new_ts.time_info = ts.time_info
    new_ts.data_info = ts.data_info
    new_ts.events = ts.events

    original_data = ts.data['data']
    
    for i, new_name in enumerate(joint_names.values()):
        new_ts.data[new_name] = original_data[:, i, :]

    return new_ts


In [16]:
# Load both motion datasets
joints_yup = np.array(json_data_local['ped_in_ped_frame'])  # 22 joints
#joints_original = np.array(json_data_local['motion'])      # 45 joints

# Take only first 22 joints from original motion to match yup format
#joints_original_22 = joints_original[:, 0:22, :]

# # Ensure both datasets have the same number of frames
min_frames = joints_yup.shape[0]
joints_yup = joints_yup[:min_frames, :, :]
#joints_original_22 = joints_original_22[:min_frames, :, :]

# Create TimeSeries for both datasets
df_joints_yup = ktk.TimeSeries(joints_yup)
#df_joints_original = ktk.TimeSeries(joints_original_22)

# Add fourth column to both
df_joints_yup_4d = add_fourth_column(df_joints_yup)
#df_joints_original_4d = add_fourth_column(df_joints_original)

# Restructure both with different naming schemes
df_joints_yup_named = restructure_timeseries(df_joints_yup_4d, joint_names)

# Create modified joint names for original motion (to avoid conflicts)
joint_names_original = {k: f"Orig_{v}" for k, v in joint_names.items()}
#df_joints_original_named = restructure_timeseries(df_joints_original_4d, joint_names_original)

# Create interconnections for Y-up motion (green skeleton)
interconnections_yup = {
    "YUp_Rightbottom": {
        "Color": [0, 1, 0],  # Green
        "Links": [["RightThigh", "RightLeg"],
                  ["RightLeg", "RightFoot"],
                  ["RightFoot", "RightToe"]],
    },
    "YUp_Leftbottom": {
        "Color": [0, 1, 0],  # Green
        "Links": [["LeftThigh", "LeftLeg"],
                  ["LeftLeg", "LeftFoot"],
                  ["LeftFoot", "LeftToe"]],
    },
    "YUp_Upperbody": {
        "Color": [0, 0.8, 0],  # Dark Green
        "Links": [["Hips", "SpineMid"],
                  ["SpineMid", "Chest"],
                  ["Chest", "Neck"],
                  ["Neck", "Head"]],
    },
    "YUp_RightTop": {
        "Color": [0, 1, 0.2],  # Light Green
        "Links": [["Chest", "RightShoulder"],
                  ["RightShoulder", "RightArm"],
                  ["RightArm", "RightForearm"],
                  ["RightForearm", "RightHand"]],
    },
    "YUp_LeftTop": {
        "Color": [0, 1, 0.2],  # Light Green
        "Links": [["Chest", "LeftShoulder"],
                  ["LeftShoulder", "LeftArm"],
                  ["LeftArm", "LeftForearm"],
                  ["LeftForearm", "LeftHand"]],
    },
    "YUp_HeadTop": {
        "Color": [0.2, 1, 0],  # Bright Green
        "Links": [["Head", "HeadTop"]],
    },
}
# Create interconnections for original motion (blue skeleton)
interconnections_original = {
    "Orig_Rightbottom": {
        "Color": [0, 0, 1],  # Blue
        "Links": [["Orig_RightThigh", "Orig_RightLeg"],
                  ["Orig_RightLeg", "Orig_RightFoot"],
                  ["Orig_RightFoot", "Orig_RightToe"]],
    },
    "Orig_Leftbottom": {
        "Color": [0, 0, 1],  # Blue
        "Links": [["Orig_LeftThigh", "Orig_LeftLeg"],
                  ["Orig_LeftLeg", "Orig_LeftFoot"],
                  ["Orig_LeftFoot", "Orig_LeftToe"]],
    },
    "Orig_Upperbody": {
        "Color": [0, 0, 0.8],  # Dark Blue
        "Links": [["Orig_Hips", "Orig_SpineMid"],
                  ["Orig_SpineMid", "Orig_Chest"],
                  ["Orig_Chest", "Orig_Neck"],
                  ["Orig_Neck", "Orig_Head"]],
    },
    "Orig_RightTop": {
        "Color": [0.2, 0, 1],  # Light Blue
        "Links": [["Orig_Chest", "Orig_RightShoulder"],
                  ["Orig_RightShoulder", "Orig_RightArm"],
                  ["Orig_RightArm", "Orig_RightForearm"],
                  ["Orig_RightForearm", "Orig_RightHand"]],
    },
    "Orig_LeftTop": {
        "Color": [0.2, 0, 1],  # Light Blue
        "Links": [["Orig_Chest", "Orig_LeftShoulder"],
                  ["Orig_LeftShoulder", "Orig_LeftArm"],
                  ["Orig_LeftArm", "Orig_LeftForearm"],
                  ["Orig_LeftForearm", "Orig_LeftHand"]],
    },
    "Orig_HeadTop": {
        "Color": [0, 0.2, 1],  # Bright Blue
        "Links": [["Orig_Head", "Orig_HeadTop"]],
    },
}

# Combine both datasets into a single TimeSeries
combined_motion_ts = df_joints_yup_named.copy()

# Add original motion data to the combined TimeSeries
# for key, value in df_joints_original_named.data.items():
#     combined_motion_ts.data[key] = value

# Process ego motion with robust error handling
#ego_motion = np.array(json_data_local['ego_motion'])
ego_in_ped_frame = np.array(json_data_local['ego_in_ped_frame'])
ego_in_ped_frame [..., 1] = 0.0
#ego_in_ped_frame = np.load('/home/mohan/Desktop/Robotics/annotation-tool/notebooks/resampled_ego.npy')

def fix_ego_motion_shape(ego_motion, target_frames):
    """Fix ego motion shape and synchronize with target frames"""
    # Handle different ego motion shapes
    if ego_motion.ndim == 1:
        # If 1D, reshape to (frames, 1) and replicate for x,y,z
        ego_motion = ego_motion.reshape(-1, 1)
        ego_motion = np.tile(ego_motion, (1, 3))  # Replicate for x,y,z
    elif ego_motion.ndim == 2 and ego_motion.shape[1] == 1:
        # If (frames, 1), replicate for x,y,z
        ego_motion = np.tile(ego_motion, (1, 3))
    elif ego_motion.ndim == 2 and ego_motion.shape[1] > 3:
        # If more than 3 columns, take only first 3
        ego_motion = ego_motion[:, :3]
    
    # Ensure we have at least 3 columns
    if ego_motion.shape[1] < 3:
        # Pad with zeros if less than 3 columns
        padding = np.zeros((ego_motion.shape[0], 3 - ego_motion.shape[1]))
        ego_motion = np.hstack([ego_motion, padding])
    
    # Trim to match target frames
    ego_motion = ego_motion[:target_frames]
    
    # If we don't have enough frames, pad with last value
    if len(ego_motion) < target_frames:
        last_frame = ego_motion[-1:] if len(ego_motion) > 0 else np.zeros((1, 3))
        padding_needed = target_frames - len(ego_motion)
        padding = np.tile(last_frame, (padding_needed, 1))
        ego_motion = np.vstack([ego_motion, padding])
    
    # Add homogeneous coordinate
    ego_motion_4d = np.column_stack([ego_motion, np.ones(len(ego_motion))])
    return ego_motion_4d

def make_ego_box_timeseries(ego_center_4d, length=5.32, width=2.13, height=1.5, y_offset=0.0):
    """
    Create eight corner points for a 3D ego box (axis-aligned) from ego centers.
    length: size along X, width: size along Z, height: size along Y, y_offset: vertical offset for the box base.
    """
    centers = ego_center_4d[:, :3]
    n = centers.shape[0]
    half_l = length / 2.0
    half_w = width / 2.0
    base_y = centers[:, 1] + y_offset
    top_y = base_y + height
    # Base rectangle
    bfl = np.column_stack([centers[:, 0] + half_l, base_y, centers[:, 2] + half_w, np.ones(n)])
    bfr = np.column_stack([centers[:, 0] + half_l, base_y, centers[:, 2] - half_w, np.ones(n)])
    bbr = np.column_stack([centers[:, 0] - half_l, base_y, centers[:, 2] - half_w, np.ones(n)])
    bbl = np.column_stack([centers[:, 0] - half_l, base_y, centers[:, 2] + half_w, np.ones(n)])
    # Top rectangle
    tfl = np.column_stack([centers[:, 0] + half_l, top_y, centers[:, 2] + half_w, np.ones(n)])
    tfr = np.column_stack([centers[:, 0] + half_l, top_y, centers[:, 2] - half_w, np.ones(n)])
    tbr = np.column_stack([centers[:, 0] - half_l, top_y, centers[:, 2] - half_w, np.ones(n)])
    tbl = np.column_stack([centers[:, 0] - half_l, top_y, centers[:, 2] + half_w, np.ones(n)])
    return {
        "EgoBoxBFL": bfl,
        "EgoBoxBFR": bfr,
        "EgoBoxBBR": bbr,
        "EgoBoxBBL": bbl,
        "EgoBoxTFL": tfl,
        "EgoBoxTFR": tfr,
        "EgoBoxTBR": tbr,
        "EgoBoxTBL": tbl,
    }

# Fix ego motion shape and add to combined TimeSeries
#ego_motion_fixed = fix_ego_motion_shape(ego_motion, len(combined_motion_ts.time))
ego_in_ped_frame_fixed = fix_ego_motion_shape(ego_in_ped_frame, len(combined_motion_ts.time))

# Add ego box corners instead of a single ego point
ego_box = make_ego_box_timeseries(ego_in_ped_frame_fixed, length=5.32, width=2.13, height=1.5, y_offset=0.0)
for key, value in ego_box.items():
    combined_motion_ts.data[key] = value

# Combine all interconnections (including ego box)
all_interconnections = {**interconnections_yup, **interconnections_original,
    "EgoBox": {
        "Color": [0.6, 0.6, 0.6],  # Grey
        "Links": [["EgoBoxBFL", "EgoBoxBFR"],
                  ["EgoBoxBFR", "EgoBoxBBR"],
                  ["EgoBoxBBR", "EgoBoxBBL"],
                  ["EgoBoxBBL", "EgoBoxBFL"],
                  ["EgoBoxTFL", "EgoBoxTFR"],
                  ["EgoBoxTFR", "EgoBoxTBR"],
                  ["EgoBoxTBR", "EgoBoxTBL"],
                  ["EgoBoxTBL", "EgoBoxTFL"],
                  ["EgoBoxBFL", "EgoBoxTFL"],
                  ["EgoBoxBFR", "EgoBoxTFR"],
                  ["EgoBoxBBR", "EgoBoxTBR"],
                  ["EgoBoxBBL", "EgoBoxTBL"]],
    },
}

# Create player with all motion data
try:
    multi_motion_player = ktk.Player(
        combined_motion_ts, 
        interconnections=all_interconnections, 
        up='y',
        background_color='w',
        track=True,
        grid_size=30,
        grid_subdivision_size=0.5,
        grid_color=(0.75, 0.75, 0.75),
        frame_size=0.3,
        default_point_color=(1,0,0),
        point_size=7,
    )
    
except Exception as e:
    # Create a version without ego motion for debugging
    debug_ts = combined_motion_ts.copy()
    if 'EgoCenter' in debug_ts.data:
        del debug_ts.data['EgoCenter']
    
    debug_player = ktk.Player(
        debug_ts, 
        interconnections={**interconnections_yup, **interconnections_original}, 
        up='y',
        track=False
    )

---

## below is for testing

In [ ]:
# joints = np.array(json_data)
# df_joints = ktk.TimeSeries(joints)
# df_joints_4d = add_fourth_column(df_joints)
# df_joints_named = restructure_timeseries(df_joints_4d, joint_names)
# interconnections = {
#     "YUp_Rightbottom": {
#         "Color": [0, 1, 0],  # Green
#         "Links": [["RightThigh", "RightLeg"],
#                   ["RightLeg", "RightFoot"],
#                   ["RightFoot", "RightToe"]],
#     },
#     "YUp_Leftbottom": {
#         "Color": [0, 1, 0],  # Green
#         "Links": [["LeftThigh", "LeftLeg"],
#                   ["LeftLeg", "LeftFoot"],
#                   ["LeftFoot", "LeftToe"]],
#     },
#     "YUp_Upperbody": {
#         "Color": [0, 0.8, 0],  # Dark Green
#         "Links": [["Hips", "SpineMid"],
#                   ["SpineMid", "Chest"],
#                   ["Chest", "Neck"],
#                   ["Neck", "Head"]],
#     },
#     "YUp_RightTop": {
#         "Color": [0, 1, 0.2],  # Light Green
#         "Links": [["Chest", "RightShoulder"],
#                   ["RightShoulder", "RightArm"],
#                   ["RightArm", "RightForearm"],
#                   ["RightForearm", "RightHand"]],
#     },
#     "YUp_LeftTop": {
#         "Color": [0, 1, 0.2],  # Light Green
#         "Links": [["Chest", "LeftShoulder"],
#                   ["LeftShoulder", "LeftArm"],
#                   ["LeftArm", "LeftForearm"],
#                   ["LeftForearm", "LeftHand"]],
#     },
#     "YUp_HeadTop": {
#         "Color": [0.2, 1, 0],  # Bright Green
#         "Links": [["Head", "HeadTop"]],
#     },
# }

# player = ktk.Player(df_joints_named, interconnections=interconnections, up='y')

In [ ]:
# ego_motion = np.array(json_data['ego_motion'])
# # ego_orientation = np.array(json_data['ego_orientation'])
# # ped_trajectory = np.array(json_data['ped_trajectory'])

# def create_ego_vehicle_timeseries(ego_motion, time_steps):
#     """
#     Create a TimeSeries object for the ego vehicle trajectory with proper dimensions
#     """
#     ego_ts = ktk.TimeSeries()
#     ego_ts.time = time_steps
    
#     # Ensure ego_motion has proper shape (frames, 4) for homogeneous coordinates
#     if ego_motion.shape[1] == 3:
#         ego_data = np.column_stack([ego_motion, np.ones(len(ego_motion))])
#     else:
#         ego_data = ego_motion
    
#     # Reshape to match expected format (frames, 1, 4) then squeeze to (frames, 4)
#     ego_ts.data['EgoCenter'] = ego_data.reshape(len(ego_data), 4)
    
#     return ego_ts

# # Create ego vehicle TimeSeries with proper time synchronization
# if len(ego_motion) == len(df_joints_names.time):
#     time_steps = df_joints_names.time
# else:
#     # Match the length of the human motion data
#     time_steps = np.linspace(0, len(ego_motion)/10.0, len(ego_motion))

# ego_ts = create_ego_vehicle_timeseries(ego_motion, time_steps)

# # Create combined TimeSeries with synchronized time
# combined_ts = df_joints_names.copy()

# # Ensure time arrays match
# if len(ego_motion) != len(combined_ts.time):
#     # Interpolate ego motion to match human motion timeline
#     from scipy.interpolate import interp1d
    
#     ego_time_original = np.linspace(0, len(ego_motion)/10.0, len(ego_motion))
#     human_time = combined_ts.time
    
#     # Interpolate each axis
#     interp_x = interp1d(ego_time_original, ego_motion[:, 0], bounds_error=False, fill_value='extrapolate')
#     interp_y = interp1d(ego_time_original, ego_motion[:, 1], bounds_error=False, fill_value='extrapolate')
#     interp_z = interp1d(ego_time_original, ego_motion[:, 2], bounds_error=False, fill_value='extrapolate')
    
#     # Create interpolated ego motion
#     ego_motion_interp = np.column_stack([
#         interp_x(human_time),
#         interp_y(human_time), 
#         interp_z(human_time),
#         np.ones(len(human_time))
#     ])
    
#     combined_ts.data['EgoCenter'] = ego_motion_interp
# else:
#     combined_ts.data['EgoCenter'] = ego_ts.data['EgoCenter']

# # Create simple interconnections (no matplotlib needed)
# combined_interconnections = interconnections.copy()

# # Create player with combined data
# combined_player = ktk.Player(combined_ts, interconnections=combined_interconnections, up='y')

In [ ]:
#ktk.Player()